# 08 · CatBoost, neural network, ExtraTrees, and the first wider ensemble

Adds three more model families, then combines everything with a stacking model that is evaluated with nested cross-validation.

**Needs:** `features2.parquet` (02), `submission.csv` + `final_oof.npy` (05), XGBoost files (07). **Writes:** `b_cat_*`, `b_mlp_*`, optional `b_et_*`, `submission_v2_lgb_lr_cat.csv`. **Time:** about 10 minutes (without ExtraTrees).

Results: CatBoost **0.7482**, neural network (scikit-learn MLP) **0.7373**. ExtraTrees takes about 13 minutes *per fold* for a weak 0.736, so we stopped it and it is left out of the default run below. Stacking everything gave only about +0.0002 over the existing blend, because the models make nearly the same errors (correlation 0.96-0.99).

**Setup.** Install the libraries once (uncomment), then set the two paths in the next cell.

In [ ]:
# %pip install -q lightgbm xgboost catboost scikit-learn pandas pyarrow scipy

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ---- EDIT THESE TWO PATHS --------------------------------------------------
DATA = os.environ.get('KAGGLE_DATA', 'data')   # folder holding the unzipped competition CSVs (train.csv, test.csv, patients.csv, ...)
WORK = os.environ.get('KAGGLE_WORK', 'work')   # folder where features, predictions and submissions are saved (shared by all notebooks)
# -----------------------------------------------------------------------------
Path(WORK).mkdir(parents=True, exist_ok=True)
print('DATA ->', Path(DATA).resolve()); print('WORK ->', Path(WORK).resolve())

In [ ]:
import pandas as pd, numpy as np, sys, time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def main(which):
    D=DATA
    drop=("hrs_","visits_","n_transfers","first_unit","last_unit","icu_","had_ed","ward_after","proc_","n_proc")
    X=pd.read_parquet(f'{WORK}/features2.parquet'); tr=pd.read_csv(f'{D}/train.csv'); te=pd.read_csv(f'{D}/test.csv')
    T=X.merge(tr,on='encounter_id'); S=te.merge(X,on='encounter_id',how='left'); y=T.readmit_30d.values
    feats=[c for c in T.columns if c not in('encounter_id','readmit_30d') and not c.startswith(drop)]
    cat=[c for c in feats if not pd.api.types.is_numeric_dtype(T[c])]; num=[c for c in feats if c not in cat and T[c].notna().sum()>0 and T[c].nunique()>1]
    folds=list(StratifiedKFold(5,shuffle=True,random_state=0).split(T,y))
    def dense():   # imputed + clipped + missing flags + one-hot (for ET / MLP)
        lo,hi=T[num].quantile(.005),T[num].quantile(.995); med=T[num].clip(lo,hi,axis=1).median()
        A=pd.concat([T[num],S[num]]); miss=A.columns[A.isna().mean()>0.01]
        def prep(df): return pd.concat([df[num].clip(lo,hi,axis=1).fillna(med),df[miss].isna().astype(int).add_suffix('_na'),df[cat].astype(object).fillna('NA')],axis=1)
        Z=pd.get_dummies(pd.concat([prep(T),prep(S)],ignore_index=True),columns=cat,dtype=float).astype(np.float32)
        return Z.iloc[:len(T)].values,Z.iloc[len(T):].values
    oof=np.zeros(len(T)); tst=np.zeros(len(S)); t0=time.time()
    if which=='cat':
        from catboost import CatBoostClassifier, Pool
        Tc=T[feats].copy(); Sc=S[feats].copy()
        for c in cat: Tc[c]=Tc[c].astype(object).fillna('NA').astype(str); Sc[c]=Sc[c].astype(object).fillna('NA').astype(str)
        for f,(a,b) in enumerate(folds):
            m=CatBoostClassifier(iterations=3000,learning_rate=0.06,depth=4,l2_leaf_reg=10,rsm=0.3,border_count=64,eval_metric='AUC',od_type='Iter',od_wait=200,thread_count=2,verbose=0,random_seed=f)
            m.fit(Pool(Tc.iloc[a],y[a],cat_features=cat),eval_set=Pool(Tc.iloc[b],y[b],cat_features=cat),use_best_model=True)
            oof[b]=m.predict_proba(Pool(Tc.iloc[b],cat_features=cat))[:,1]; tst+=m.predict_proba(Pool(Sc,cat_features=cat))[:,1]/5
            print(which,f,m.get_best_iteration(),'%.5f'%roc_auc_score(y[b],oof[b]),'t=%.0f'%(time.time()-t0),flush=True)
    else:
        Zt,Zs=dense()
        if which=='et':
            from sklearn.ensemble import ExtraTreesClassifier
            for f,(a,b) in enumerate(folds):
                m=ExtraTreesClassifier(400,min_samples_leaf=40,max_features=0.25,n_jobs=2,random_state=f).fit(Zt[a],y[a])
                oof[b]=m.predict_proba(Zt[b])[:,1]; tst+=m.predict_proba(Zs)[:,1]/5
                print(which,f,'%.5f'%roc_auc_score(y[b],oof[b]),'t=%.0f'%(time.time()-t0),flush=True)
        if which=='mlp':
            from sklearn.neural_network import MLPClassifier
            from sklearn.preprocessing import StandardScaler
            for f,(a,b) in enumerate(folds):
                sc=StandardScaler().fit(Zt[a]); xa=np.clip(sc.transform(Zt[a]),-6,6); xb=np.clip(sc.transform(Zt[b]),-6,6); xs=np.clip(sc.transform(Zs),-6,6)
                for seed in range(3):
                    m=MLPClassifier((64,16),alpha=3.0,learning_rate_init=1e-3,batch_size=512,early_stopping=True,validation_fraction=0.1,n_iter_no_change=6,max_iter=60,random_state=seed*10+f).fit(xa,y[a])
                    oof[b]+=m.predict_proba(xb)[:,1]/3; tst+=m.predict_proba(xs)[:,1]/15
                print(which,f,'%.5f'%roc_auc_score(y[b],oof[b]),'t=%.0f'%(time.time()-t0),flush=True)
    print(which,'OOF AUC %.5f'%roc_auc_score(y,oof))
    np.save(f'{WORK}/b_{which}_oof.npy',oof); np.save(f'{WORK}/b_{which}_test.npy',tst); print('done')

In [ ]:
for which in ['mlp', 'cat']:        # add 'et' only if you have an hour to spare
    main(which)

### 8.1 Stacking all available models (nested cross-validation)

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from scipy.optimize import minimize
W=WORK
X=pd.read_parquet(f'{W}/features2.parquet',columns=['encounter_id']); tr=pd.read_csv(f'{DATA}/train.csv'); te=pd.read_csv(f'{DATA}/test.csv')
y=X.merge(tr,on='encounter_id').readmit_30d.values
lg=lambda p:np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))
sub=pd.read_csv(f'{W}/submission.csv'); assert sub.encounter_id.tolist()==te.encounter_id.tolist()
M={'lgb_lr_blend':(np.load(f'{W}/final_oof.npy'),sub.readmit_30d.values),
   'xgb':(np.load(f'{W}/xgb_oof_single.npy'),np.load(f'{W}/xgb_test_single.npy')),
   'xgb_bag':(np.load(f'{W}/xgb_oof_bag.npy'),np.load(f'{W}/xgb_test_bag.npy'))}
for k in ['cat','et','mlp']:
    try: M[k]=(np.load(f'{W}/b_{k}_oof.npy'),np.load(f'{W}/b_{k}_test.npy'))
    except Exception as e: print('missing',k)
names=list(M); O=np.column_stack([lg(M[k][0]) for k in names]); Ts=np.column_stack([lg(M[k][1]) for k in names])
A=lambda s:roc_auc_score(y,s)
print('Individual OOF AUC:'); [print(f'  {k:14s} {A(O[:,i]):.5f}') for i,k in enumerate(names)]
print('OOF logit correlation:'); print(pd.DataFrame(O,columns=names).corr().round(3))
res={}
res['mean_all']=(O.mean(1),Ts.mean(1))
# weighted blend: maximise AUC surrogate = logistic loss on stacked logits (non-negative weights), evaluated by nested CV
def fitw(Oa,ya):
    f=lambda w:-np.sum(ya*(Oa@w)-np.log1p(np.exp(Oa@w)))+1e-3*np.sum(w**2)
    r=minimize(f,np.ones(Oa.shape[1])/Oa.shape[1],bounds=[(0,None)]*Oa.shape[1],method='L-BFGS-B'); return r.x
nest=np.zeros(len(y))
for a,b in StratifiedKFold(5,shuffle=True,random_state=42).split(O,y): nest[b]=O[b]@fitw(O[a],y[a])
res['weighted_nested_cv']=(nest,Ts@fitw(O,y)); w=fitw(O,y); print('Stacking weights:',dict(zip(names,w.round(3))))
# meta logistic regression stacker (with intercept) nested
nest2=np.zeros(len(y))
for a,b in StratifiedKFold(5,shuffle=True,random_state=42).split(O,y): nest2[b]=LogisticRegression(C=1.0).fit(O[a],y[a]).decision_function(O[b])
res['logreg_stack_nested_cv']=(nest2,LogisticRegression(C=1.0).fit(O,y).decision_function(Ts))
print('\nEnsemble OOF AUC:'); [print(f'  {k:24s} {A(v[0]):.5f}') for k,v in res.items()]
print(f"  {'current submission':24s} {A(O[:,0]):.5f}")
# bootstrap paired comparison vs current
rng=np.random.RandomState(0); n=len(y); keys=list(res); d={k:[] for k in keys}
for _ in range(300):
    i=rng.randint(0,n,n); base=roc_auc_score(y[i],O[i,0])
    for k in keys: d[k].append(roc_auc_score(y[i],res[k][0][i])-base)
for k in keys: v=np.array(d[k]); print(f'{k:24s} minus current: {v.mean():+.5f}  95% CI [{np.percentile(v,2.5):+.5f}, {np.percentile(v,97.5):+.5f}]')
best=max(keys,key=lambda k:np.mean(d[k])); print('best:',best)
import pickle; pickle.dump((res,names,w),open(f'{W}/ens_res.pkl','wb'))

### 8.2 Optional second submission: current blend (81%) + CatBoost (19%)
The weights come from the stacking result above.

In [ ]:
import pandas as pd
def validate_submission(path):
    """Checks a file against sample_submission.csv: same ids in the same order, no gaps, probabilities in [0, 1]."""
    s = pd.read_csv(path); ss = pd.read_csv(f'{DATA}/sample_submission.csv')
    ok = s.encounter_id.tolist() == ss.encounter_id.tolist()
    print(Path(path).name, '| rows', len(s), '| ids match sample:', ok, '| NaN:', int(s.readmit_30d.isna().sum()),
          '| all in [0,1]:', bool(s.readmit_30d.between(0, 1).all()), '| mean %.4f' % s.readmit_30d.mean())
    assert ok and s.readmit_30d.notna().all() and s.readmit_30d.between(0, 1).all()

In [ ]:
import numpy as np, pandas as pd
lg = lambda p: np.log(np.clip(p, 1e-9, 1 - 1e-9) / (1 - np.clip(p, 1e-9, 1 - 1e-9)))
te = pd.read_csv(f'{DATA}/test.csv'); cur = pd.read_csv(f'{WORK}/submission.csv'); cat = np.load(f'{WORK}/b_cat_test.npy')
z = 0.809 * lg(cur.readmit_30d.values) + 0.192 * lg(cat); p = 1 / (1 + np.exp(-z))
pd.DataFrame({'encounter_id': te.encounter_id, 'readmit_30d': p.round(6)}).to_csv(f'{WORK}/submission_v2_lgb_lr_cat.csv', index=False)
validate_submission(f'{WORK}/submission_v2_lgb_lr_cat.csv')